## 1. Importing Basic Libraries and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("data/train.csv")
pd.set_option('display.max_columns', 100)
df.head()

## 2. Data Exploration , Cleaning and Handling

In [ ]:
df.shape

# Rows : 1,03,904      Columns : 25

In [ ]:
df.info()

In [ ]:
# 1. Drop unnecessary columns
df = df.drop(['Unnamed: 0', 'id'], axis=1, errors='ignore')

# 2. Clean up header-row duplicates (if any)
df = df[df['Age'] != 'Age']

# 3. Convert numeric columns safely
# This targets columns that look like numbers and converts them, 
# while leaving categorical columns (strings) untouched.
for col in df.columns:
    # Try to convert each column; if it fails, it stays as is
    try:
        df[col] = df[col].astype(float) # pd.to_numeric(df[col], errors='ignore')
        
    except:
        pass

In [ ]:
df.info()
# print(df['Flight Distance'].unique().tolist(), end='')
# print(df['Flight Distance'].dtype)
# # df['Flight Distance'] = pd.to_numeric(df['Flight Distance'], errors='ignore')
# df['Flight Distance']= df['Flight Distance'].astype(float)
# print(df['Flight Distance'].dtype)

In [ ]:
len(df.columns)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
# Convert everything possible to numeric
numeric_df = df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')

# Now check if it's still empty
if numeric_df.empty:
    print("Error: No numeric data found even after conversion!")
else:
    corr = numeric_df.corr()
    sns.heatmap(corr, cmap="Blues")
    plt.show()

In [ ]:
# Departure Delay and Arrival Delay :  highly positively correlated -- If a flight departs late, it is likely to arrive late as well

In [ ]:
df.describe()

#### Insights

- Arrival Delay Column:
  * The minimum arrival delay is 0 (indicating no delay).
  * The maximum arrival delay is 1584.
- Flight Distance Column:
  * The shortest flight distance in the dataset is 31(suspicious)
  * The longest flight distance in the dataset is 4983.
- Departure Delay Column:
  * The minimum departure delay is 0.
  * The maximum departure delay is 1592.
- Note:
  * There appears to be outliers in the data as there is a significant difference between the third quartile (Q3) and the maximum value.

In [ ]:
print(numerical.shape)
numerical.head(2)

In [ ]:
### Check for Outliers

numerical = df.select_dtypes(include=['int', 'float'])
blue_palette = sns.color_palette("Blues", n_colors=len(numerical.columns))

fig, axes = plt.subplots(6, 3, figsize=(50, 60))
axes = axes.flatten()

for i, col in enumerate(numerical.columns):
    sns.boxplot(x=df[col], ax=axes[i], color=blue_palette[i])
    axes[i].set_title(col, fontsize=30)
    
plt.show()

In [ ]:
### We know these columns have outliers : 

# 1. Departure Delay ( too much )
# 2. Arrival Delay  (  too much )
# 3. Check-In service ( very less )
# 4. Flight Distance ( Moderate )

In [ ]:
### Handle Outliers

def handle_outliers(df, columns):
    for column in columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[column] = df[column].clip(lower=lower_bound, upper=upper_bound)

    return df

columns_to_handel= ['Flight Distance','Departure Delay in Minutes','Arrival Delay in Minutes', 'Checkin service']
df = handle_outliers(df, columns_to_handel)

In [ ]:
### Handle Null values of Arrival Delay in Minutes

df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Arrival Delay in Minutes'].median())


In [ ]:
df.isnull().sum()

## 3. EDA

In [ ]:
new_df = df.copy()

In [ ]:
new_df.columns

In [ ]:
columns_with_six_categories = new_df.columns[new_df.nunique() == 6]
columns_with_six_categories

In [ ]:
new_df["Cleanliness"].value_counts()

In [ ]:
ordinal_mapping = {
    0: "Very Poor",
    1: "Poor",
    2: "Average",
    3: "Good",
    4: "Excellent",
    5: "Outstanding"
}

In [ ]:
new_df[columns_with_six_categories] = new_df[columns_with_six_categories].map(lambda x: ordinal_mapping.get(x, x))

In [ ]:
new_df.head(1)

In [ ]:
df.head(1)

In [ ]:
df["Baggage handling"].value_counts()

# It does not have any 0 value thats why we need another mapping for it

In [ ]:
mapping = {
    0: "Very Poor",
    1: "Poor",
    2: "Average",
    3: "Good",
    4: "Excellent",
}

new_df['Baggage handling'] = new_df['Baggage handling'].apply(lambda x: ordinal_mapping.get(x, x))

In [ ]:
new_df.head(1)

In [ ]:
df["Checkin service"].value_counts()

# It also has other type of mapping

In [ ]:
mapping = {
    1.5: "Very Poor",
    2.0: "Poor",
    3.0: "Average",
    4.0: "Good",
    5.0: "Excellent",
}

new_df['Checkin service'] = new_df['Checkin service'].apply(lambda x: ordinal_mapping.get(x, x))

In [ ]:
new_df.head(1)

### Now alll data is set for EDA

In [ ]:
df.columns

### UNIVARIATE EDA

In [ ]:
numerical_columns = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
categorical_columns = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 
                       'Inflight wifi service', 'Departure/Arrival time convenient', 
                       'Ease of Online booking', 'Gate location', 'Food and drink', 
                       'Online boarding', 'Seat comfort', 'Inflight entertainment', 
                       'On-board service', 'Leg room service', 'Baggage handling', 
                       'Checkin service', 'Inflight service', 'Cleanliness']


In [ ]:
fig, axes = plt.subplots(nrows=8, ncols=3, figsize=(24, 24))
axes = axes.flatten()

for i, col in enumerate(numerical_columns):
    sns.histplot(df[col], kde=True, ax=axes[i], color='blue')
    axes[i].set_title(f'Histogram of {col}')

# Plot bar plots for categorical features
for j, col in enumerate(categorical_columns):
    if i + 1 + j < len(axes):
        sns.countplot(x=df[col], ax=axes[i + 1 + j], palette='viridis')
        axes[i + 1 + j].set_title(f'Count Plot of {col}')

plt.tight_layout()
plt.show()

## Bivariate EDA 

In [ ]:
target_column = 'satisfaction'

# Numerical vs. Categorical
for col in numerical_columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(x=df[target_column], y=df[col], palette='viridis')
    plt.title(f'Boxplot of {col} by {target_column}')
    plt.xticks(rotation=45)
    plt.show()

# Categorical vs. Categorical
for col in categorical_columns:
    plt.figure(figsize=(12, 6))
    sns.countplot(x=df[col], hue=df[target_column], palette='viridis')
    plt.title(f'Count Plot of {col} by {target_column}')
    plt.xticks(rotation=45)
    plt.show()
    
# Categorical vs. Numerical (Bar plots with average values)
for col in categorical_columns:
    plt.figure(figsize=(12, 6))
    sns.barplot(x=df[col], y=df[numerical_columns[0]], hue=df[target_column], palette='viridis', ci=None)
    plt.title(f'Bar Plot of {numerical_columns[0]} by {col} and {target_column}')
    plt.xticks(rotation=45)
    plt.show()

### Multivariate EDA

In [ ]:
sns.pairplot(df[numerical_columns + [target_column]], hue=target_column, palette='viridis')
plt.suptitle('Pairplot of Numerical Features', y=1.02)
plt.show()

## IMPORTANT INSIGHTS

* Majority of people who travel are from Age-group 20-40 . This age group is of youngsters like students , working professionals , tourists , buisness man, etc.
* Most of people only tavel for upto 1000km distance. As only tourist people travel for long distances.
* Most of flights are not delayed but if they are delayed they are delayed for more than 24 hours.Gnerally flights are delayed due to extreme weather conditions and if weather is bad it generally remains bad for quite a time.
* Female travels slightly more than Males.
* Most of flight customers are loyal which means they have taken thsi flight previously too.
* Most of people travel for buisness purposes.
* Most people travel in Buisness and Eco Class . Eco Plus price is generally between these two classes and just offfers some basic amenties like leg-room , etc which many people dont find valuable enough.
* Flight wifi service is average . They should make it better.
* They should focus on their online booking technolgy as people are facing problem due to this.
* Seat Comfort and Inflight service is also not that good.

## 4. FEATURE ENGINEERING

In [ ]:
### Make a new column using Feature Construction ( Domain Knowledge )

df['Total Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']

In [ ]:
df['Delay Ratio'] = df['Total Delay'] / (df['Flight Distance'] + 1)

In [ ]:
df.head(1)

In [ ]:
### Convert Age using Binning
df['Age Group'] = pd.cut(df['Age'], bins=[0, 18, 30, 50, 100], labels=['Child', 'Youngster', 'Adult', 'Senior'])

In [ ]:
df.head(1)

In [ ]:
df.dtypes

In [ ]:
### Label Encoding

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

columns_to_encode = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction', 'Age Group']

label_mappings = {}

for col in columns_to_encode:
    df[col] = le.fit_transform(df[col])
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))


for col, mapping in label_mappings.items():
    print(f"Mapping for {col}: {mapping}")

In [ ]:
df.columns

In [ ]:
df.head(1)

In [ ]:
### Feature Selection

from sklearn.model_selection import train_test_split

X = df.drop(columns='satisfaction')
y = df['satisfaction']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

# Calculate mutual information
mutual_info = mutual_info_classif(X_train, y_train, discrete_features=True)

# Create a DataFrame for mutual information
mutual_info_df = pd.DataFrame({
    'Feature': X.columns,
    'Mutual Information': mutual_info
}).sort_values(by='Mutual Information', ascending=False)

print(mutual_info_df)

In [ ]:
#### Create our final dataframe with important features

top_features = mutual_info_df.head(12)['Feature'].tolist()

In [ ]:
final_df = df[top_features + ['satisfaction']]

In [ ]:
final_df.head(3)

In [ ]:
# Checking Imbalanced Data
final_df["satisfaction"].value_counts()

## 5. Model Training

In [ ]:
final_df.columns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
import xgboost as xgb

# Sample 10% of the data
df_sample = final_df.sample(frac=0.05, random_state=42)

# Prepare the sample data
X_sample = final_df.drop(columns='satisfaction')
y_sample = final_df['satisfaction']


# Split the sampled data
X_train_sample, X_test_sample, y_train_sample, y_test_sample = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

# Initialize classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(n_estimators=50, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=50),
    'AdaBoost': AdaBoostClassifier(n_estimators=50),
    'Support Vector Classifier': SVC(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(),
    'LightGBM': lgb.LGBMClassifier(),
    'XGBoost': xgb.XGBClassifier(eval_metric='mlogloss')
}

# Train and evaluate each classifier
results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_sample, y_train_sample)
    y_pred = clf.predict(X_test_sample)
    accuracy = accuracy_score(y_test_sample, y_pred)
    results[name] = accuracy

# Print results
for name, accuracy in results.items():
    print(f"{name}: {accuracy:.4f}")


In [ ]:
### We know now LGBM is the best model for us 

In [ ]:
### Making LGBM MODEL

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import lightgbm as lgb

In [ ]:
X = final_df.drop(columns='satisfaction')
y = final_df['satisfaction']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
lgbm_model = lgb.LGBMClassifier()
lgbm_model.fit(X_train, y_train)

In [ ]:
y_pred = lgbm_model.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Use weighted average for multi-class
recall = recall_score(y_test, y_pred, average='weighted') 
f1 = f1_score(y_test, y_pred, average='weighted')  

In [ ]:
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

In [ ]:
### We dont do hyperparamter tuning as there was a risk of overfitting as model is already performing quite good

## 6. MODEL SAVING

In [ ]:
import pickle

with open('lgbm_model.pkl', 'wb') as file:
    pickle.dump(lgbm_model, file)


## 7. MODEL LOADING AND TESTING

In [ ]:
with open('lgbm_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

In [ ]:
X_train[8:15]

In [ ]:
y_train[8:15]

In [ ]:
X_new = np.array([
    [5 , 0.000000 ,4 ,0, 0,	2 ,3736.5, 5, 2, 2,	4,	3],
    [2	, 0.000000 , 	2 , 1	, 1 ,3	,986.0 ,	3 ,	4 ,	4	, 2	, 3]
])

In [ ]:
y_pred_new = loaded_model.predict(X_new)

In [ ]:

print("Predictions:", y_pred_new)


In [ ]:
### It is doing pretty good predictions..........................

### WE HAVE OUR ML MODEL NOW

### Let's Go to APP Building

In [ ]:
# |---- | ----   |
# | Age | Gender |
# | 10  | M      |

df.show()

In [ ]:
a, b, c= 1, 2, 3
print(a)
print(b)
print(c)

In [ ]:
_, _, x= 10, 20, 30
print(_)
print(_)
print(x)

In [1]:
from src.paths_config import *

In [2]:
TEST_DATA_PATH

'./artifact/ingested_data/test.csv'